In [ ]:
#Trenowanie modelu SVM na cechach HOG
import os
import cv2
import numpy as np
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

#KONFIGURACJA
data_dir = '../data/selected_signs'
IMG_SIZE = (64, 64) # Ujednolicony rozmiar dla wszystkich zdjęć

X = [] # Wektory cech HOG
y = [] # Etykiety (nazwy klas)

print("Przetwarzanie zdjęć i ekstrakcja HOG...")

# Przechodzimy przez każdy folder
for class_name in os.listdir(data_dir):
    class_path = os.path.join(data_dir, class_name)
    
    if os.path.isdir(class_path):
        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)
            
            # Wczytanie obrazu
            img = cv2.imread(img_path)
            if img is not None:
                # 1. Zmiana rozmiaru
                img_resized = cv2.resize(img, IMG_SIZE)
                
                # 2. Konwersja do skali szarości
                gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
                
                # 3. Ekstrakcja cech HOG
                features = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                               cells_per_block=(2, 2), block_norm='L2-Hys',
                               visualize=False)
                
                X.append(features)
                y.append(class_name)

# Konwersja do tablic numpy dla scikit-learn
X = np.array(X)
y = np.array(y)

print(f"Przetworzono {len(X)} zdjęć.")
print(f"Długość pojedynczego wektora HOG: {X.shape[1]}")

#TRENING MODELU
print("\nDzielenie danych na zbiór treningowy (80%) i testowy (20%)...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Trenuję klasyfikator SVM...")
svm_model = SVC(kernel='linear', probability=True)
svm_model.fit(X_train, y_train)

#EWALUACJA
print("Testuję model...")
y_pred = svm_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nOgólna skuteczność modelu: {accuracy * 100:.2f}%\n")

print("Raport klasyfikacji:")
print(classification_report(y_test, y_pred))